In [6]:
# Instala dependencias usadas en el notebook
%pip install -U azure-ai-inference
print("Dependencias instaladas: azure-ai-inference")

  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas: azure-ai-inference



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3) Modelos Multimodales
### Objetivo: 
Desplegar un modelo multimodal y probar interacciones que involucren imágenes, audio y/o texto combinado (por ejemplo: describir una imagen, transcribir audio y responder preguntas sobre su contenido, etc.).

[Documentación](https://learn.microsoft.com/en-us/azure/foundry-classic/foundry-models/how-to/use-chat-multi-modal?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=programming-language-python)


### Imagen: explicación
En este bloque se hace una petición multimodal con texto + imagen usando un modelo Phi multimodal. Se pide una descripción y detalles concretos de la escena.

In [20]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.identity import DefaultAzureCredential
from azure.ai.inference.models import SystemMessage, UserMessage, TextContentItem, ImageContentItem, ImageUrl

endpoint = "https://foundrymario2.services.ai.azure.com/models"
model_deployment = "Phi-4-multimodal-instruct"  

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
    credential_scopes=["https://ai.azure.com/.default"],
 )

image_url = "https://st1.uvnimg.com/cc/39/520d2ecb48bfaa0fffa73a835f8f/GettyImages-476215942.jpg"

response = client.complete(
    model=model_deployment,
    messages=[
        SystemMessage(
            "Eres un analista visual riguroso. No inventes datos: si algo no se ve claramente, indicalo."
        ),
        UserMessage(content=[
            TextContentItem(
                text=(
                    "Analiza la imagen y responde en este formato exacto:\n"
                    "1) Descripcion general (3-5 lineas).\n"
                    "2) Evidencias visuales clave (lista de 4-6 puntos).\n"
                    "3) Año probable de la escena (si es visible): <año o rango>.\n"
                    "4) Trofeo que están alzando (si es visible): <nombre del trofeo o 'no visible'>.\n"
                    "5) Nivel de confianza del año: alto/medio/bajo.\n"
                    "6) Colores de las camisetas."
                )
            ),
            ImageContentItem(image_url=ImageUrl(url=image_url))
        ]),
    ],
    temperature=0.1,
    max_tokens=700,
 )

print(f"{response.choices[0].message.role}: {response.choices[0].message.content}")
print("Model:", response.model)
print("Usage:")
print("\tPrompt tokens:", response.usage.prompt_tokens)
print("\tCompletion tokens:", response.usage.completion_tokens)
print("\tTotal tokens:", response.usage.total_tokens)

ChatRole.ASSISTANT: 1) La imagen muestra a un equipo de fútbol celebrando con un trofeo.
2) Evidencias visuales clave:
   - Los jugadores están celebrando.
   - Hay un trofeo grande en la mano de uno de los jugadores.
   - Los jugadores están usando camisetas de un equipo de fútbol.
   - Hay medallas alrededor de la cintura de algunos jugadores.
3) Año probable de la escena: 2015.
4) Trofeo que están alzando: UEFA Champions League.
5) Nivel de confianza del año: alto.
6) Colores de las camisetas: azul y rojo.
Model: vision
Usage:
	Prompt tokens: 1930
	Completion tokens: 126
	Total tokens: 2056


### Audio: explicación
En este bloque se carga un archivo de audio local, se transcribe y después se analiza el texto resultante con el modelo de chat.

In [25]:
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    TextContentItem,
    AudioContentItem,
    InputAudio,
    AudioContentFormat,
    SystemMessage,
    UserMessage
)
from azure.identity import DefaultAzureCredential

endpoint = "https://foundrymario2.services.ai.azure.com/models"
model_deployment = "Phi-4-multimodal-instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
    credential_scopes=["https://cognitiveservices.azure.com/.default"],
)

response = client.complete(
    model=model_deployment,
    messages=[
        SystemMessage("You are an AI assistant for translating and transcribing audio clips."),
        UserMessage(
            [
                TextContentItem(text="Please translate this audio snippet to spanish."),
                AudioContentItem(
                    input_audio=InputAudio.load(
                        audio_file="hello_how_are_you.mp3",
                        audio_format=AudioContentFormat.MP3
                    )
                ),
            ],
        ),
    ],
)

print(f"{response.choices[0].message.role}: {response.choices[0].message.content}")
print("Model:", response.model)
print("Usage:")
print("\tPrompt tokens:", response.usage.prompt_tokens)
print("\tCompletion tokens:", response.usage.completion_tokens)
print("\tTotal tokens:", response.usage.total_tokens)

ChatRole.ASSISTANT: Hola, ¿Cómo estás
Model: speech
Usage:
	Prompt tokens: 77
	Completion tokens: 6
	Total tokens: 83





### Entregable: 
Notebook con llamadas al endpoint multimodal mostrando varios ejemplos: subida/consulta de imágenes, audios y prompts mixtos; incluir control de formatos y manejo de respuestas (texto y/o estructuras).

---

Formato y criterios de entrega
- Cada parte debe entregarse como un notebook `.ipynb` autocontenido que incluya:
	- Sección de configuración / credenciales (explicando cómo configurar variables de entorno localmente).
	- Código reproducible conecta a un modelo ya desplegado, realiza las llamadas y procesa las respuestas.
	- Celdas de explicación y resultados visibles (salidas, figuras, JSON validados).
	- Una sección final de conclusiones y problemas encontrados.






 ### Salida estructurada y validación

En este bloque se le pide al modelo que devuelva JSON válido con claves fijas. Después se valida que la salida pueda parsearse y que contenga todos los campos esperados antes de mostrarla formateada.


In [26]:
import json

expected_keys = [
    "descripcion_general",
    "evidencias_visuales",
    "ano_probable",
    "trofeo",
    "nivel_confianza",
    "colores_camisetas",
]

structured_response = client.complete(
    model=model_deployment,
    messages=[
        SystemMessage(
            "Eres un analista visual riguroso. Responde SOLO con JSON valido, sin markdown ni texto adicional."
        ),
        UserMessage(
            content=[
                TextContentItem(
                    text=(
                        "Analiza la imagen y devuelve exactamente este JSON:\n"
                        '{\n'
                        '  "descripcion_general": "texto breve",\n'
                        '  "evidencias_visuales": ["punto 1", "punto 2"],\n'
                        '  "ano_probable": "año o rango",\n'
                        '  "trofeo": "nombre o no visible",\n'
                        '  "nivel_confianza": "alto|medio|bajo",\n'
                        '  "colores_camisetas": ["color 1", "color 2"]\n'
                        '}\n\n'
                        "Si un dato no se ve con claridad, usa 'no visible' o una lista vacía."
                    )
                ),
                ImageContentItem(image_url=ImageUrl(url=image_url)),
            ]
        ),
    ],
    temperature=0.1,
    max_tokens=700,
)

raw_output = structured_response.choices[0].message.content.strip()
print("Salida cruda:")
print(raw_output)
print()

json_text = raw_output
if json_text.startswith("```"):
    first_brace = json_text.find("{")
    last_brace = json_text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        json_text = json_text[first_brace:last_brace + 1]

try:
    parsed_output = json.loads(json_text)
    missing_keys = [key for key in expected_keys if key not in parsed_output]
    type_errors = []

    if "descripcion_general" in parsed_output and not isinstance(parsed_output["descripcion_general"], str):
        type_errors.append("descripcion_general debe ser texto")
    if "evidencias_visuales" in parsed_output and not isinstance(parsed_output["evidencias_visuales"], list):
        type_errors.append("evidencias_visuales debe ser una lista")
    if "ano_probable" in parsed_output and not isinstance(parsed_output["ano_probable"], str):
        type_errors.append("ano_probable debe ser texto")
    if "trofeo" in parsed_output and not isinstance(parsed_output["trofeo"], str):
        type_errors.append("trofeo debe ser texto")
    if "nivel_confianza" in parsed_output and parsed_output["nivel_confianza"] not in {"alto", "medio", "bajo"}:
        type_errors.append("nivel_confianza debe ser alto, medio o bajo")
    if "colores_camisetas" in parsed_output and not isinstance(parsed_output["colores_camisetas"], list):
        type_errors.append("colores_camisetas debe ser una lista")

    if missing_keys or type_errors:
        print("Validacion: JSON valido, pero hay problemas:")
        if missing_keys:
            print("- Claves faltantes:", missing_keys)
        if type_errors:
            print("- Errores de tipo/valor:", type_errors)
    else:
        print("Validacion: JSON valido y estructura correcta.")
        print(json.dumps(parsed_output, ensure_ascii=False, indent=2))
except json.JSONDecodeError as exc:
    print("Validacion: la respuesta no es JSON valido.")
    print(exc)


Salida cruda:
{
  "descripcion_general": "jugadores de fútbol celebrando",
  "evidencias_visuales": ["trophy", "camisetas"],
  "ano_probable": "2015",
  "trofeo": "UEFA Champions League",
  "nivel_confianza": "alto",
  "colores_camisetas": ["blue", "red"]
}

Validacion: JSON valido y estructura correcta.
{
  "descripcion_general": "jugadores de fútbol celebrando",
  "evidencias_visuales": [
    "trophy",
    "camisetas"
  ],
  "ano_probable": "2015",
  "trofeo": "UEFA Champions League",
  "nivel_confianza": "alto",
  "colores_camisetas": [
    "blue",
    "red"
  ]
}
